In [ ]:
!pip install transformers datasets accelerate scikit-learn sentencepiece protobuf -q

In [ ]:
from google.colab import drive
import os, zipfile

drive.mount('/content/drive')

zip_path = "/content/drive/MyDrive/deberta-v3-small.zip"
cache_dir = os.path.expanduser("~/.cache/huggingface/hub/")
os.makedirs(cache_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as z:
    for info in z.infolist():
        fixed_name = info.filename.replace("\\", "/")
        out_path = os.path.join(cache_dir, fixed_name)
        if info.is_dir() or fixed_name.endswith("/"):
            os.makedirs(out_path, exist_ok=True)
        elif info.file_size == 0:
            os.makedirs(os.path.dirname(out_path), exist_ok=True)
        else:
            os.makedirs(os.path.dirname(out_path), exist_ok=True)
            with z.open(info) as src, open(out_path, 'wb') as dst:
                dst.write(src.read())

model_cache = os.path.join(cache_dir, "models--microsoft--deberta-v3-small")
print(f"Model extracted to: {model_cache}")
print(f"Contents: {os.listdir(model_cache)}")

Mounted at /content/drive
Model extracted to: /root/.cache/huggingface/hub/models--microsoft--deberta-v3-small
Contents: ['refs', 'snapshots']


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import json
import os
import gc
import re
import random
import shutil

os.environ["HF_HUB_OFFLINE"] = "1"

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Using device: cuda
GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB


In [ ]:
df_raw = pd.read_csv('ExioNAICS.csv')

SECTOR_MERGE = {
    '31': '31-33', '32': '31-33', '33': '31-33',
    '44': '44-45', '45': '44-45',
    '48': '48-49', '49': '48-49',
}

naics2_raw = df_raw[['NAICS_2 Code', 'NAICS_2 Title', 'NAICS_2 Description']].drop_duplicates(subset='NAICS_2 Code').copy()
naics2_raw['NAICS_2 Code'] = naics2_raw['NAICS_2 Code'].astype(str)
naics2_raw['sector_code'] = naics2_raw['NAICS_2 Code'].map(SECTOR_MERGE).fillna(naics2_raw['NAICS_2 Code'])

naics2_corpus = naics2_raw.drop_duplicates(subset='sector_code').copy()
naics2_corpus = naics2_corpus.sort_values('sector_code').reset_index(drop=True)
naics2_corpus['corpus_text'] = naics2_corpus['NAICS_2 Title'] + '. ' + naics2_corpus['NAICS_2 Description'].fillna('')

corpus_texts = naics2_corpus['corpus_text'].tolist()
sector_codes = naics2_corpus['sector_code'].tolist()
sector_titles = naics2_corpus['NAICS_2 Title'].tolist()
code_to_idx = {code: i for i, code in enumerate(sector_codes)}
NUM_CLASSES = len(corpus_texts)

print(f"Merged sectors (1-to-1 mapping):")
for i, (code, title) in enumerate(zip(sector_codes, sector_titles)):
    print(f"  {i:>2}: {code:>5} -> {title}")

df = pd.read_csv('ExioNAICS_preprocessed.csv')
df['NAICS Code'] = df['NAICS Code'].astype(str)
df['naics2_raw'] = df['NAICS Code'].str[:2]
df['sector'] = df['naics2_raw'].map(SECTOR_MERGE).fillna(df['naics2_raw'])
df['naics2_idx'] = df['sector'].map(code_to_idx)

missing = df['naics2_idx'].isna().sum()
if missing > 0:
    print(f"\nDropping {missing} samples with unmapped sector codes")
    df = df.dropna(subset=['naics2_idx']).reset_index(drop=True)
df['naics2_idx'] = df['naics2_idx'].astype(int)

print(f"\nNAICS-2 sectors: {NUM_CLASSES}")
print(f"Dataset: {len(df)} samples")
print(f"\nSamples per sector:")
print(df['sector'].value_counts().sort_index().to_string())


def preprocess_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


print("\n=== Building NAICS-6 augmentation training data ===")

naics_sources = [
    ('NAICS-6', 'NAICS Code', 'NAICS Title', 'Description'),
    ('NAICS-5', 'NAICS_5 Code', 'NAICS_5 Title', 'NAICS_5 Description'),
    ('NAICS-4', 'NAICS_4 Code', 'NAICS_4 Title', 'NAICS_4 Description'),
    ('NAICS-3', 'NAICS_3 Code', 'NAICS_3 Title', 'NAICS_3 Description'),
]

aug_rows = []
for level, code_col, title_col, desc_col in naics_sources:
    sub = df_raw[[code_col, title_col, desc_col]].drop_duplicates(subset=code_col).dropna(subset=[code_col])
    for _, row in sub.iterrows():
        code = str(row[code_col])
        title = str(row[title_col]) if not pd.isna(row[title_col]) else ""
        desc = str(row[desc_col]) if not pd.isna(row[desc_col]) else ""
        text = (title + ". " + desc).strip()
        if len(text) < 20:
            continue
        naics2_prefix = code[:2]
        sector = SECTOR_MERGE.get(naics2_prefix, naics2_prefix)
        if sector not in code_to_idx:
            continue
        aug_rows.append({
            'naics_code': code,
            'level': level,
            'clean_description': preprocess_text(text),
            'sector': sector,
            'naics2_idx': code_to_idx[sector],
        })

df_aug = pd.DataFrame(aug_rows).drop_duplicates(subset='naics_code').reset_index(drop=True)
print(f"Levels used: {df_aug['level'].value_counts().to_dict()}")

print(f"Augmentation samples: {len(df_aug)}")
print(f"\nSamples per sector (augmentation):")
print(df_aug['sector'].value_counts().sort_index().to_string())

del df_raw
gc.collect()

NAICS-2 sectors: 24
Codes: ['11', '21', '22', '23', '31', '32', '33', '42', '44', '45', '48', '49', '51', '52', '53', '54', '55', '56', '61', '62', '71', '72', '81', '92']
Dataset: 20535 samples

Samples per sector:
naics2
11     645
21     459
22     152
23     740
31    1476
32    1993
33    3950
42    1980
44     996
45     682
48     746
49     134
51     696
52     790
53     524
54    1060
55      70
56     692
61     259
62     700
71     357
72     264
81     768
92     402


481

In [ ]:
MODEL_NAME = "microsoft/deberta-v3-small"
MAX_LENGTH = 192
BATCH_SIZE = 8
NUM_EPOCHS = 30
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
SEED = 42
VAL_RATIO = 0.1
PATIENCE = 5

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print(f"=== NAICS-2 Cross-Encoder (DeBERTa-v3-small) ===")
print(f"  Model:        {MODEL_NAME}")
print(f"  Max length:   {MAX_LENGTH}")
print(f"  Batch size:   {BATCH_SIZE} queries x {NUM_CLASSES} candidates = {BATCH_SIZE * NUM_CLASSES} seq/step")
print(f"  Epochs:       {NUM_EPOCHS}")
print(f"  LR:           {LEARNING_RATE}")
print(f"  Patience:     {PATIENCE}")

=== NAICS-2 Cross-Encoder (DeBERTa-v3-small) ===
  Model:        microsoft/deberta-v3-small
  Max length:   192
  Batch size:   8 queries x 24 candidates = 192 seq/step
  Epochs:       30
  LR:           2e-05
  Patience:     5


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=1, torch_dtype=torch.float32
).to(device)

print(f"Total params: {sum(p.numel() for p in model.parameters()):,}")

test_enc = tokenizer("company description", "naics sector description", return_tensors="pt",
                     max_length=MAX_LENGTH, truncation=True, padding=True)
test_enc = {k: v.to(device) for k, v in test_enc.items() if k in ['input_ids', 'attention_mask']}
with torch.no_grad():
    test_out = model(**test_enc)
print(f"Output shape: {test_out.logits.shape}")
print(f"NaN check: {torch.isnan(test_out.logits).any().item()}")

The tokenizer you are loading from 'microsoft/deberta-v3-small' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias       

Total params: 141,895,681
Output shape: torch.Size([1, 1])
NaN check: False


In [ ]:
def pretokenize(queries, num_classes, max_length, label_name):
    ids, masks = [], []
    for i, q in enumerate(queries):
        pairs_ids, pairs_masks = [], []
        for c_idx in range(num_classes):
            enc = tokenizer(
                q, corpus_texts[c_idx],
                max_length=max_length, truncation=True,
                padding='max_length', return_tensors='pt'
            )
            pairs_ids.append(enc['input_ids'].squeeze(0))
            pairs_masks.append(enc['attention_mask'].squeeze(0))
        ids.append(torch.stack(pairs_ids))
        masks.append(torch.stack(pairs_masks))
        if (i + 1) % 2000 == 0:
            print(f"  [{label_name}] {i+1}/{len(queries)} done")
    return torch.stack(ids), torch.stack(masks)


print(f"Pre-tokenizing {len(df)} real samples x {NUM_CLASSES} candidates...")
all_input_ids, all_attention_masks = pretokenize(
    df['clean_description'].tolist(), NUM_CLASSES, MAX_LENGTH, "real"
)
all_labels = torch.tensor(df['naics2_idx'].values, dtype=torch.long)
print(f"Real pre-tokenized shape: {all_input_ids.shape}")

print(f"\nPre-tokenizing {len(df_aug)} augmentation samples x {NUM_CLASSES} candidates...")
aug_input_ids, aug_attention_masks = pretokenize(
    df_aug['clean_description'].tolist(), NUM_CLASSES, MAX_LENGTH, "aug"
)
aug_labels = torch.tensor(df_aug['naics2_idx'].values, dtype=torch.long)
print(f"Aug pre-tokenized shape: {aug_input_ids.shape}")

total_mem = (all_input_ids.nbytes + all_attention_masks.nbytes + aug_input_ids.nbytes + aug_attention_masks.nbytes) / 1e9
print(f"\nTotal memory: {total_mem:.2f} GB")


class PreTokenizedDataset(Dataset):
    def __init__(self, input_ids, attention_masks, labels):
        self.input_ids = input_ids
        self.attention_masks = attention_masks
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_masks[idx],
            'label': self.labels[idx],
        }

print("Pre-tokenization complete.")

Pre-tokenizing all 20535 samples x 24 candidates...
This runs once and makes training ~10x faster.

  5000/20535 done
  10000/20535 done
  15000/20535 done
  20000/20535 done

Pre-tokenized shape: torch.Size([20535, 24, 192])
Memory: 1.51 GB
Pre-tokenization complete.


In [ ]:
train_idx, val_idx = train_test_split(
    np.arange(len(df)), test_size=VAL_RATIO, random_state=SEED, stratify=df['naics2_idx']
)

train_input_ids = torch.cat([all_input_ids[train_idx], aug_input_ids], dim=0)
train_attention_masks = torch.cat([all_attention_masks[train_idx], aug_attention_masks], dim=0)
train_labels = torch.cat([all_labels[train_idx], aug_labels], dim=0)

train_dataset = PreTokenizedDataset(train_input_ids, train_attention_masks, train_labels)

val_descriptions = df['clean_description'].iloc[val_idx].tolist()
val_labels = df['naics2_idx'].iloc[val_idx].tolist()

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)

print(f"Real training samples:  {len(train_idx)}")
print(f"Augmentation samples:   {len(df_aug)}")
print(f"Total training samples: {len(train_dataset)}  ({len(train_loader)} batches)")
print(f"Val samples:            {len(val_descriptions)}  (real only, no augmentation)")
print(f"\nClass distribution (combined training):")
train_counts = pd.Series(train_labels.numpy()).value_counts().sort_index()
print(f"  Min: {train_counts.min()}, Max: {train_counts.max()}, Mean: {train_counts.mean():.0f}")

Train: 18481 samples, 2311 batches
Val:   2054 samples
Class distribution (train):
  Min: 63, Max: 3555, Mean: 770


In [ ]:
@torch.no_grad()
def evaluate(model, tokenizer, descriptions, labels, corpus_texts, max_length, score_batch=64):
    model.eval()
    n = len(descriptions)
    num_classes = len(corpus_texts)
    top1, top3, top5 = 0, 0, 0
    all_preds = []
    all_labels = labels

    for i in range(n):
        query = descriptions[i]
        true_label = labels[i]
        scores = []

        for j in range(0, num_classes, score_batch):
            batch_docs = corpus_texts[j:j+score_batch]
            enc = tokenizer(
                [query] * len(batch_docs), batch_docs,
                max_length=max_length, truncation=True, padding=True,
                return_tensors='pt'
            )
            enc = {k: v.to(device) for k, v in enc.items() if k in ['input_ids', 'attention_mask']}
            logits = model(**enc).logits.squeeze(-1)
            scores.extend(logits.cpu().tolist())

        scores_t = torch.tensor(scores)
        topk = scores_t.topk(min(5, num_classes)).indices.tolist()
        all_preds.append(topk[0])

        if topk[0] == true_label: top1 += 1
        if true_label in topk[:3]: top3 += 1
        if true_label in topk[:5]: top5 += 1

    from sklearn.metrics import f1_score, classification_report
    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    weighted_f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)

    return {
        'top1': top1 / n, 'top3': top3 / n, 'top5': top5 / n,
        'macro_f1': macro_f1, 'weighted_f1': weighted_f1,
        'all_preds': all_preds,
    }

print("Evaluation function defined")

Evaluation function defined


In [ ]:
from transformers import get_cosine_schedule_with_warmup

CHECKPOINT_DIR = "/content/drive/MyDrive/naics2_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs("results_naics2", exist_ok=True)

LABEL_SMOOTHING = 0.05

label_counts = np.bincount(train_labels.numpy(), minlength=NUM_CLASSES).astype(np.float64)
class_weights_np = 1.0 / np.maximum(label_counts, 1.0)
class_weights_np = class_weights_np / class_weights_np.sum() * NUM_CLASSES
class_weights = torch.tensor(class_weights_np, dtype=torch.float, device=device)

print("Class weights (inverse frequency, normalized to mean=1):")
for i, (code, title, cnt, w) in enumerate(zip(sector_codes, sector_titles, label_counts, class_weights_np)):
    print(f"  {code:>5} ({int(cnt):>5} samples)  weight={w:.3f}  {title}")
print(f"\nLabel smoothing: {LABEL_SMOOTHING}")

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

best_top1 = 0.0
best_epoch = 0
patience_counter = 0
epoch_log = []
start_epoch = 1

checkpoint_path = os.path.join(CHECKPOINT_DIR, "latest_checkpoint.pt")
if os.path.exists(checkpoint_path):
    print("Found checkpoint on Google Drive, resuming...")
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    start_epoch = ckpt['epoch'] + 1
    best_top1 = ckpt['best_top1']
    best_epoch = ckpt['best_epoch']
    patience_counter = ckpt['patience_counter']
    epoch_log = ckpt['epoch_log']
    print(f"  Resumed from epoch {ckpt['epoch']}, best Top-1: {best_top1:.4f} (epoch {best_epoch})")
    print(f"  Patience: {patience_counter}/{PATIENCE}")
    del ckpt
    torch.cuda.empty_cache()
else:
    print("No checkpoint found, starting from scratch.")

print(f"\nTotal steps: {total_steps}")
print(f"Warmup: {warmup_steps} steps")
print(f"Training epochs {start_epoch} to {NUM_EPOCHS}")
print(f"\n{'='*80}")
print(f"{'Ep':>3} {'TrLoss':>8} {'Top1':>7} {'Top3':>7} {'Top5':>7} {'MacF1':>7} {'WtF1':>7} {'LR':>10}")
print(f"{'='*80}")

for epoch in range(start_epoch, NUM_EPOCHS + 1):
    model.train()
    total_loss = 0.0
    num_batches = 0

    for batch in train_loader:
        bsz = batch['input_ids'].size(0)
        num_cands = batch['input_ids'].size(1)

        input_ids = batch['input_ids'].view(-1, MAX_LENGTH).to(device)
        attention_mask = batch['attention_mask'].view(-1, MAX_LENGTH).to(device)
        labels = batch['label'].to(device)

        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits.squeeze(-1)
        scores = logits.view(bsz, num_cands)
        loss = F.cross_entropy(scores, labels, weight=class_weights, label_smoothing=LABEL_SMOOTHING)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        num_batches += 1

    avg_loss = total_loss / num_batches
    lr = scheduler.get_last_lr()[0]

    metrics = evaluate(model, tokenizer, val_descriptions, val_labels, corpus_texts, MAX_LENGTH)

    epoch_log.append({
        'epoch': epoch, 'train_loss': avg_loss, 'lr': lr,
        'top1': metrics['top1'], 'top3': metrics['top3'], 'top5': metrics['top5'],
        'macro_f1': metrics['macro_f1'], 'weighted_f1': metrics['weighted_f1'],
    })

    print(f"{epoch:>3} {avg_loss:>8.4f} {metrics['top1']:>7.4f} {metrics['top3']:>7.4f} {metrics['top5']:>7.4f} "
          f"{metrics['macro_f1']:>7.4f} {metrics['weighted_f1']:>7.4f} {lr:>10.2e}")

    if metrics['top1'] > best_top1:
        best_top1 = metrics['top1']
        best_epoch = epoch
        patience_counter = 0
        torch.save(model.state_dict(), "results_naics2/best_model.pt")
        shutil.copy("results_naics2/best_model.pt", os.path.join(CHECKPOINT_DIR, "best_model.pt"))
    else:
        patience_counter += 1

    pd.DataFrame(epoch_log).to_csv("results_naics2/naics2_epoch_log.csv", index=False)

    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_top1': best_top1,
        'best_epoch': best_epoch,
        'patience_counter': patience_counter,
        'epoch_log': epoch_log,
    }, checkpoint_path)

    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}")
        break

print(f"\n{'='*80}")
print(f"Best Top-1: {best_top1:.4f} at epoch {best_epoch}")

best_path = os.path.join(CHECKPOINT_DIR, "best_model.pt")
if os.path.exists(best_path):
    model.load_state_dict(torch.load(best_path, weights_only=True))
else:
    model.load_state_dict(torch.load("results_naics2/best_model.pt", weights_only=True))
print("Loaded best model.")

No checkpoint found, starting from scratch.

Total steps: 69330
Warmup: 6933 steps
Training epochs 1 to 30

 Ep   TrLoss    Top1    Top3    Top5   MacF1    WtF1         LR
  1   2.4975  0.4192  0.7093  0.8301  0.3866  0.3738   6.67e-06
  2   1.8492  0.4581  0.7376  0.8505  0.4364  0.4129   1.33e-05
  3   1.7368  0.4703  0.7425  0.8549  0.4491  0.4276   2.00e-05
  4   1.6277  0.4460  0.7454  0.8583  0.4383  0.4175   1.99e-05
  5   1.4968  0.4737  0.7590  0.8612  0.4450  0.4299   1.97e-05
  6   1.3822  0.4684  0.7483  0.8578  0.4439  0.4264   1.94e-05


In [ ]:
from sklearn.metrics import classification_report

print("=== Final Evaluation on Validation Set ===\n")
final_metrics = evaluate(model, tokenizer, val_descriptions, val_labels, corpus_texts, MAX_LENGTH)

print(f"  Top-1 Accuracy:  {final_metrics['top1']:.4f}")
print(f"  Top-3 Accuracy:  {final_metrics['top3']:.4f}")
print(f"  Top-5 Accuracy:  {final_metrics['top5']:.4f}")
print(f"  Macro F1:        {final_metrics['macro_f1']:.4f}")
print(f"  Weighted F1:     {final_metrics['weighted_f1']:.4f}")
print(f"  Best epoch:      {best_epoch}")

target_names = [f"{sector_codes[i]} - {sector_titles[i]}" for i in range(NUM_CLASSES)]
print(f"\n=== Per-Class Report ===\n")
print(classification_report(val_labels, final_metrics['all_preds'], target_names=target_names, zero_division=0))

results = {
    "method": "DeBERTa-v3-small Cross-Encoder (NAICS-2, all candidates scored)",
    "config": {
        "model": MODEL_NAME, "max_length": MAX_LENGTH, "batch_size": BATCH_SIZE,
        "num_classes": NUM_CLASSES, "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY, "num_epochs_trained": best_epoch,
        "class_weighted_loss": True, "label_smoothing": LABEL_SMOOTHING,
        "naics6_augmentation": True,
    },
    "results": {
        "top1": final_metrics['top1'], "top3": final_metrics['top3'], "top5": final_metrics['top5'],
        "macro_f1": final_metrics['macro_f1'], "weighted_f1": final_metrics['weighted_f1'],
    },
}

with open("results_naics2/naics2_results.json", "w") as f:
    json.dump(results, f, indent=2, default=str)

import zipfile, glob
with zipfile.ZipFile("results_naics2.zip", "w") as zf:
    for f in glob.glob("results_naics2/*.json") + glob.glob("results_naics2/*.csv"):
        zf.write(f)
        print(f"  Added: {f}")

from google.colab import files
files.download("results_naics2.zip")
print("\nDownloaded results_naics2.zip")